# 05 — Components

Reusable composite structures with `@component`.

**What you learn:**
- `@component`: decorator for handlers with a body
- `main_tag='div'`: declares the DOM tag for parent validation
- The handler receives a fresh Bag (`comp`) and populates it
- Components are expanded lazily at build time
- Reuse: call the component multiple times with different parameters

**Important:** On `HtmlBuilder`, custom tags are always `@component` (not `@element`). Elements are the 112 HTML5 primitives. User extensions expand into real HTML tags.

**Prerequisites:** 04_pointers

In [ ]:
from IPython.display import HTML
from genro_builders.builder import component
from genro_builders.contrib.html import HtmlBuilder
from genro_builders.manager import BuilderManager


class CatalogBuilder(HtmlBuilder):
    """HtmlBuilder extended with a product_card component."""

    @component(main_tag='div')
    def product_card(self, comp, name=None, price=None, description=None, **kwargs):
        """main_tag='div' tells the validator: treat me as a div."""
        card = comp.div(_class='card')
        card.h3(name or 'Unnamed')
        card.p(price or '', _class='price')
        card.p(description or '')

In [ ]:
class ProductCatalog(BuilderManager):
    def __init__(self):
        self.page = self.set_builder('page', CatalogBuilder)
        self.run()

    def store(self, data):
        data['products'] = [
            {'name': 'Wireless Headphones', 'price': '$79.99', 'description': 'Premium sound, 30h battery.'},
            {'name': 'Mechanical Keyboard', 'price': '$129.99', 'description': 'Cherry MX, RGB backlight.'},
            {'name': 'USB-C Hub', 'price': '$49.99', 'description': '7-in-1: HDMI, USB-A, SD, ethernet.'},
        ]

    def main(self, source):
        source.head().style('''
            body { font-family: sans-serif; max-width: 700px; margin: 0 auto; }
            .catalog { display: grid; grid-template-columns: repeat(auto-fill, minmax(200px, 1fr)); gap: 1em; }
            .card { border: 1px solid #ddd; border-radius: 8px; padding: 1em; }
            .card h3 { margin-top: 0; }
            .price { color: #16a34a; font-weight: bold; font-size: 1.2em; }
        ''')
        body = source.body()
        body.h1('Product Catalog')
        catalog = body.div(_class='catalog')
        for p in self.reactive_store['products']:
            catalog.product_card(name=p['name'], price=p['price'], description=p['description'])

app = ProductCatalog()
HTML(app.page.render())